# OCI IAM Playground

Explore and manage OCI Identity & Access Management:
- List users and their details
- List groups and memberships
- Add/remove users from groups

## 1. Setup

In [1]:
import os
import oci
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / '.env')

config     = oci.config.from_file()
identity   = oci.identity.IdentityClient(config)
TENANCY_ID = os.environ['TENANCY_OCID']

print('OCI Identity client ready ✓')

OCI Identity client ready ✓


## 2. List Users

In [2]:
users = oci.pagination.list_call_get_all_results(
    identity.list_users,
    compartment_id=TENANCY_ID,
).data

users_df = pd.DataFrame([{
    'name':          u.name,
    'email':         u.email or '-',
    'state':         u.lifecycle_state,
    'mfa':           u.is_mfa_activated,
    'created':       str(u.time_created)[:10],
    'id':            u.id,
} for u in users])

print(f'Total users: {len(users_df)}')
users_df[['name', 'email', 'state', 'mfa', 'created']]

Total users: 2


,name,email,state,mfa,created
0,abouradanis@gmail.com,abouradanis@gmail.com,ACTIVE,True,2025-02-26
1,testbiuser,abouradanis@gmaill.com,ACTIVE,False,2026-04-28


## 3. List Groups & Memberships

In [3]:
groups = oci.pagination.list_call_get_all_results(
    identity.list_groups,
    compartment_id=TENANCY_ID,
).data

# Build memberships
rows = []
for g in groups:
    members = oci.pagination.list_call_get_all_results(
        identity.list_user_group_memberships,
        compartment_id=TENANCY_ID,
        group_id=g.id,
    ).data
    user_names = [
        u.name for u in users
        if u.id in {m.user_id for m in members}
    ]
    rows.append({
        'group':       g.name,
        'description': g.description or '-',
        'state':       g.lifecycle_state,
        'created':     str(g.time_created)[:10],
        'members':     ', '.join(user_names) if user_names else '(none)',
        'id':          g.id,
    })

groups_df = pd.DataFrame(rows)
print(f'Total groups: {len(groups_df)}')
groups_df[['group', 'description', 'state', 'members', 'created']]

Total groups: 3


,group,description,state,members,created
0,Administrators,Administrators,ACTIVE,abouradanis@gmail.com,2025-02-26
1,ds_group,-,ACTIVE,abouradanis@gmail.com,2026-01-19
2,All Domain Users,A group representing all users.,ACTIVE,(none),2025-02-26


## 4. Add User to Group

Look up user and group by name, then add the membership.

In [5]:
TARGET_USER  = 'testbiuser'   # change as needed
TARGET_GROUP = 'ds_group'     # change as needed

# Look up OCIDs by name
user_row  = users_df[users_df['name'].str.lower() == TARGET_USER.lower()]
group_row = groups_df[groups_df['group'].str.lower() == TARGET_GROUP.lower()]

if user_row.empty:
    print(f'User "{TARGET_USER}" not found. Available users:')
    print(users_df['name'].tolist())
elif group_row.empty:
    print(f'Group "{TARGET_GROUP}" not found. Available groups:')
    print(groups_df['group'].tolist())
else:
    user_id  = user_row.iloc[0]['id']
    group_id = group_row.iloc[0]['id']

    try:
        identity.add_user_to_group(
            oci.identity.models.AddUserToGroupDetails(
                user_id=user_id,
                group_id=group_id,
            )
        )
        print(f'✓ {TARGET_USER} added to {TARGET_GROUP}')
    except oci.exceptions.ServiceError as e:
        if e.status == 409:
            print(f'{TARGET_USER} is already a member of {TARGET_GROUP}')
        else:
            raise

✓ testbiuser added to ds_group
